# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*What am I trying to predict, and why is this the right ML task?*

### The problem

The goal of this lane is to **prioritize content pages that may need human review**.

Instead of automatically deciding that a page should be refreshed, the model gives each page a **priority score**. Pages with higher scores can be reviewed first by the SEO/content team.

This makes the problem a **supervised priority-scoring task**:

- **Input:** historical page and search-performance signals
- **Output:** probability that a page will experience a meaningful traffic decline
- **Business use:** rank pages so limited human review time can be spent on the most promising candidates

### Target definition

For Week 5, I use a future-looking target based on May performance:

`decline = (may_clicks < 0.8 × april_clicks)`

A page is therefore labelled as:

- `1` → May clicks were **20% or more lower** than April clicks
- `0` → May clicks did not fall by 20% or more

The eligibility rules are:

- `impressions_total >= 1000`
- `april_clicks >= 10`

This target is different from the earlier starter-data proxy used in Weeks 2–4. Here, the model is evaluated against the **actual May outcome**.

### Why ranking matters

The goal is not simply to classify every page correctly.

In practice, the content team has limited time, so the more useful question is:

> **If we can review only 50 pages, which 50 should we look at first?**

Because of this, the main evaluation metric is **Precision@50**.

Precision@50 tells us how many of the top 50 pages ranked by the model were actually declining pages.

### Why compare against a baseline?

Before using more complex ML models, I keep a simple rule-based baseline:

`april_clicks < march_clicks`

Flagged pages are ranked using April impressions.

This gives me a simple reference point to beat. If a more complex model cannot improve on this transparent rule, there is little reason to prefer the added complexity.

### Method choice

I compare three learned models against the baseline:

- **Logistic Regression** — simple and interpretable
- **Decision Tree** — captures basic non-linear relationships
- **Random Forest** — combines multiple decision trees and can capture more complex patterns

The models are compared using the **same data, same validation design, and same Precision@K metrics**.

The final model is selected based on evidence from the validation results, rather than choosing the most complex model automatically.

### Decision point and leakage

The model should only use information that would have been available **before May**.

Therefore, May clicks and impressions are used to create the target but are **not available as model inputs**.

This is important because using future information would cause **data leakage** and make the evaluation unrealistically optimistic.

### Section 1 takeaway

The task is therefore:

**Historical page signals → predict May decline risk → rank pages → prioritize human review.**

The model is a **decision-support tool**, not an automatic content-refresh decision maker.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the training dataset

*Before training any model, I first made sure the data, target, features, and validation setup were correct.*

### What data did I use?

For Week 5, I used the **real FlyRank warehouse data** rather than creating or estimating historical values.

I worked with daily search-performance data from **February 1 to May 31, 2026** and aggregated it to one row per content page.

The final dataset contained:

- **39,308,592** daily rows
- **407,121** unique content pages
- **70** clients
- **16,513** eligible pages after applying the assignment rules
- **36** clients represented in the eligible set

### Creating the target

The goal is to identify pages that experienced a meaningful decline in May.

I defined the target as:

`decline = (may_clicks < 0.8 × april_clicks)`

So:

- `1` = May clicks were at least **20% lower** than April clicks
- `0` = May clicks did not fall by 20% or more

I also applied the eligibility rules:

- `impressions_total >= 1000`
- `april_clicks >= 10`

After filtering, the target distribution was:

- **9,640 pages → target 0**
- **6,873 pages → target 1**
- **41.62% decline rate**

### Features available before May

I created **9 features** using information available before the May decision point:

1. `impressions_total`
2. `clicks_total`
3. `april_impressions`
4. `april_clicks`
5. `feb_clicks`
6. `momentum`
7. `ctr`
8. `active_days`
9. `weighted_position`

These features describe the page's historical search performance without using the future May outcome.

### Preventing leakage

A key requirement was making sure the model could not see the answer in advance.

Therefore:

- **May clicks and impressions** were used only to create the target.
- They were **not used as model features**.
- Client and content IDs were used for grouping and joining, but **not as predictive features**.
- All 9 model features had **0% missing values** in the final eligible dataset.

This keeps the modeling setup aligned with the real decision point: *what could we have known before May?*

### Validation setup

I used **5-fold GroupKFold by client**.

This means that pages from the same client are kept together, so the model is tested on **clients it did not train on**.

The final validation check showed:

- **5 folds**
- **0 client overlap** between training and validation sets

This is important because randomly splitting pages could allow the model to learn client-specific patterns and make the results look better than they really are.

### Section 2 takeaway

At the end of this step, I had a clean modeling dataset with:

- a clearly defined future target,
- 9 leakage-safe historical features,
- 16,513 eligible pages,
- and a client-grouped validation design.

With this setup fixed, I could move to Section 3 and compare different models against the baseline fairly.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from huggingface_hub import snapshot_download
from sklearn.model_selection import GroupKFold

print('==================================================')
print('SECTION 2: RAM-OPTIMIZED WAREHOUSE AGGREGATION & SPLIT DESIGN')
print('==================================================')

# 1. Obtain HF_TOKEN to access gated FlyRank/internship-warehouse dataset
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    env_file = Path('.env')
    if not env_file.exists():
        env_file = Path('../.env')
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip()

if not HF_TOKEN:
    print('[WARNING] HF_TOKEN not found in environment or .env file.')
    print('Please provide your Hugging Face READ token to stream real warehouse daily partitions.')
    raise ValueError('HF_TOKEN required to load real gated warehouse dataset FlyRank/internship-warehouse.')

# 2. Download Feb-May 2026 daily performance partitions from Hugging Face
local_dir = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    allow_patterns=[
        'dim_*.parquet',
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
        'fact_content_daily_performance/month=2026-04/*.parquet',
        'fact_content_daily_performance/month=2026-05/*.parquet'
    ],
    token=HF_TOKEN
)

fact_files = list(Path(local_dir).glob('fact_content_daily_performance/**/*.parquet'))
print(f'Found {len(fact_files)} daily parquet partition files.')

# 3. Construct RAM-Efficient Polars Lazy Scan using actual warehouse column names
# Real warehouse columns: gsc_clicks, gsc_impressions, gsc_avg_position
lazy_daily = pl.scan_parquet(fact_files).select([
    pl.col('report_date').cast(pl.Utf8),
    pl.col('client_hash_id').cast(pl.Categorical),
    pl.col('content_hash_id').cast(pl.Categorical),
    pl.col('gsc_clicks').cast(pl.Int32),
    pl.col('gsc_impressions').cast(pl.Int32),
    pl.col('gsc_avg_position').cast(pl.Float32)
])

# Memory-efficient lazy duplicate check at daily grain
dup_check = lazy_daily.group_by(['report_date', 'client_hash_id', 'content_hash_id']).len().filter(pl.col('len') > 1).select(pl.len()).collect()
dup_count = dup_check[0, 0] if len(dup_check) > 0 else 0
print(f'1. Grain check (report_date x client_hash_id x content_hash_id): duplicates = {dup_count}')
assert dup_count == 0, 'Duplicate rows detected at daily grain!'

# 4. Streamed Lazy Aggregation across Feb, Mar, Apr, and May
feb_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-02-28')
mar_mask = (pl.col('report_date') >= '2026-03-01') & (pl.col('report_date') <= '2026-03-31')
apr_mask = (pl.col('report_date') >= '2026-04-01') & (pl.col('report_date') <= '2026-04-30')
pre_may_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-04-30')
may_mask = (pl.col('report_date') >= '2026-05-01') & (pl.col('report_date') <= '2026-05-31')

# Stream main monthly aggregations using gsc_clicks and gsc_impressions
lazy_main_agg = lazy_daily.group_by(['client_hash_id', 'content_hash_id']).agg([
    pl.col('gsc_clicks').filter(feb_mask).sum().alias('feb_clicks'),
    pl.col('gsc_impressions').filter(feb_mask).sum().alias('feb_impressions'),

    pl.col('gsc_clicks').filter(mar_mask).sum().alias('march_clicks'),
    pl.col('gsc_impressions').filter(mar_mask).sum().alias('march_impressions'),

    pl.col('gsc_clicks').filter(apr_mask).sum().alias('april_clicks'),
    pl.col('gsc_impressions').filter(apr_mask).sum().alias('april_impressions'),

    pl.col('gsc_impressions').filter(pre_may_mask).sum().alias('impressions_total'),
    pl.col('gsc_clicks').filter(pre_may_mask).sum().alias('clicks_total'),

    pl.col('report_date').filter(pre_may_mask & (pl.col('gsc_impressions') > 0)).n_unique().alias('active_days'),

    pl.col('gsc_clicks').filter(may_mask).sum().alias('may_clicks'),
    pl.col('gsc_impressions').filter(may_mask).sum().alias('may_impressions')
])

# Stream weighted_position calculation excluding gsc_avg_position == 0
lazy_pos_agg = lazy_daily.filter(pre_may_mask & (pl.col('gsc_avg_position') > 0)).group_by(['client_hash_id', 'content_hash_id']).agg([
    (pl.col('gsc_avg_position') * pl.col('gsc_impressions')).sum().alias('pos_num'),
    pl.col('gsc_impressions').sum().alias('pos_den')
])

# Collect small per-page tables using Polars streaming engine
agg_main = lazy_main_agg.collect()
agg_pos = lazy_pos_agg.collect()

# Join small per-page summary tables
agg_df = agg_main.join(agg_pos, on=['client_hash_id', 'content_hash_id'], how='left')
agg_df = agg_df.with_columns([
    (pl.col('pos_num') / pl.col('pos_den')).alias('weighted_position')
]).drop(['pos_num', 'pos_den'])

# Free memory
del agg_main, agg_pos
gc.collect()

# 5. Derived Features & Actual May Target
agg_df = agg_df.with_columns([
    (pl.col('april_clicks') / (pl.col('feb_clicks') + 1.0)).alias('momentum'),
    ((pl.col('clicks_total') / pl.col('impressions_total')) * 100).alias('ctr'),
    # Actual Week 5 Target: decline = (may_clicks < 0.8 * april_clicks).astype(int)
    (pl.col('may_clicks') < (0.8 * pl.col('april_clicks'))).cast(pl.Int64).alias('decline')
])

# 6. Apply Eligibility Filter & Enforce Explicit Deterministic Sorting
elig_df = agg_df.filter((pl.col('impressions_total') >= 1000) & (pl.col('april_clicks') >= 10))
# Deterministic sort by client_hash_id and content_hash_id
elig_df = elig_df.sort(['client_hash_id', 'content_hash_id'])

print('\n2. Real Warehouse Dataset & Eligibility Report:')
print(f'Total aggregated content pages: {len(agg_df):,}')
print(f'Total eligible pages:           {len(elig_df):,} ({len(elig_df)/len(agg_df)*100:.2f}% retained)')
print(f'Distinct eligible clients:     {elig_df["client_hash_id"].n_unique()}')
print(f'Date ranges used:                Feb 1, 2026 - Apr 30, 2026 (Features) | May 1 - May 31, 2026 (Target)')

target_counts = elig_df['decline'].to_pandas().value_counts().to_dict()
base_rate = elig_df['decline'].mean()
print(f'Target 0/1 counts:              {target_counts}')
print(f'Target Base Rate (decline %):   {base_rate * 100:.2f}%')

feature_cols = [
    'impressions_total',
    'clicks_total',
    'april_impressions',
    'april_clicks',
    'feb_clicks',
    'momentum',
    'ctr',
    'active_days',
    'weighted_position'
]

print(f'\nFeatures ({len(feature_cols)} total):')
print(feature_cols)

print('\nFeature Missingness in Eligible Dataset:')
print({col: int(elig_df[col].null_count()) for col in feature_cols})

elig_pd = elig_df.to_pandas()
elig_pd['weighted_position'] = elig_pd['weighted_position'].fillna(elig_pd['weighted_position'].median())

# 7. Execute 5-Fold GroupKFold Split & Client Overlap Checks
print('\n3. 5-Fold GroupKFold Client Overlap Checks:')
gkf = GroupKFold(n_splits=5)
X_data = elig_pd[feature_cols]
y_data = elig_pd['decline']
groups_data = elig_pd['client_hash_id']

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_data, y_data, groups_data)):
    tr_clients = set(groups_data.iloc[tr_idx])
    val_clients = set(groups_data.iloc[val_idx])
    overlap = tr_clients.intersection(val_clients)
    assert len(overlap) == 0, f'Fold {fold} HAS CLIENT OVERLAP!'
    print(f'Fold {fold}: Train Rows={len(tr_idx):4d} | Val Rows={len(val_idx):4d} | Train Clients={len(tr_clients):2d} | Val Clients={len(val_clients):2d} | Overlap={len(overlap)}')

print('\nZERO CLIENT LEAKAGE: All 5 folds passed zero-overlap assertions.')


SECTION 2: RAM-OPTIMIZED WAREHOUSE AGGREGATION & SPLIT DESIGN


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Found 4 daily parquet partition files.
1. Grain check (report_date x client_hash_id x content_hash_id): duplicates = 0

2. Real Warehouse Dataset & Eligibility Report:
Total aggregated content pages: 407,121
Total eligible pages:           16,513 (4.06% retained)
Distinct eligible clients:     36
Date ranges used:                Feb 1, 2026 - Apr 30, 2026 (Features) | May 1 - May 31, 2026 (Target)
Target 0/1 counts:              {0: 9640, 1: 6873}
Target Base Rate (decline %):   41.62%

Features (9 total):
['impressions_total', 'clicks_total', 'april_impressions', 'april_clicks', 'feb_clicks', 'momentum', 'ctr', 'active_days', 'weighted_position']

Feature Missingness in Eligible Dataset:
{'impressions_total': 0, 'clicks_total': 0, 'april_impressions': 0, 'april_clicks': 0, 'feb_clicks': 0, 'momentum': 0, 'ctr': 0, 'active_days': 0, 'weighted_position': 0}

3. 5-Fold GroupKFold Client Overlap Checks:
Fold 0: Train Rows=12205 | Val Rows=4308 | Train Clients=35 | Val Clients= 1 | Overlap

## 3. Train + compare vs my baseline

*Same data, same metric, and same client-grouped validation setup as the baseline.*

### Models I tested

I compared the simple Week 5 baseline with three learned models:

- **Baseline:** Flags pages where `april_clicks < march_clicks` and ranks flagged pages by April impressions.
- **Logistic Regression:** A simple and interpretable linear model.
- **Decision Tree:** A shallow tree (`max_depth=3`) that can capture simple non-linear patterns.
- **Random Forest:** An ensemble of decision trees that can capture more complex relationships.

All models were evaluated on the same **16,513 eligible pages** using the same **5-fold GroupKFold split by client**.

### Leakage and ranking checks

The models use only the **9 pre-May features** created in Section 2.

May clicks and impressions are used only to create the actual target and are **never used as model features**.

For each fold:

1. The model is trained only on the training clients.
2. It predicts the unseen validation clients.
3. Pages are ranked by predicted decline probability.
4. Ties are handled deterministically.

The main metric is **Precision@50**, with Precision@10, Precision@20, and Precision@100 also reported.

### Model Performance

| Model | Precision@10 | Precision@20 | Precision@50 | Precision@100 |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline** | 0.4000 | 0.3700 | 0.3920 | 0.3880 |
| **Logistic Regression** | 0.3800 | 0.3700 | 0.4240 | 0.4380 |
| **Decision Tree** | 0.4000 | 0.3900 | 0.3240 | 0.3460 |
| **Random Forest** | **0.4600** | **0.4300** | **0.4440** | **0.4480** |

### What the results show

**Random Forest performed best at the primary Precision@50 metric**, with a score of **0.4440**.

The baseline achieved **0.3920**, so Random Forest improved Precision@50 by **5.2 percentage points**.

In practical terms, this means that across the validation folds, the Random Forest prioritized about **22 actual declining pages out of every 50 reviewed**, compared with about **20 out of 50** for the baseline.

The improvement suggests that combining multiple historical signals can produce a better review ranking than the simple rule.

However, this does **not** guarantee that refreshing a page will improve its future SEO performance.

### Fold-by-Fold Precision@50

| Fold | Baseline | Logistic Regression | Decision Tree | Random Forest |
| :--- | :---: | :---: | :---: | :---: |
| Fold 0 | 0.3200 | 0.3800 | 0.3600 | 0.4400 |
| Fold 1 | 0.3800 | 0.3800 | 0.3600 | 0.3200 |
| Fold 2 | 0.4200 | 0.4200 | 0.3600 | 0.4600 |
| Fold 3 | 0.4200 | 0.5600 | 0.4800 | 0.6400 |
| Fold 4 | 0.4400 | 0.3600 | 0.4200 | 0.3600 |
| **Mean** | **0.3920** | **0.4240** | **0.3240** | **0.4440** |

The fold results also show that Random Forest was **not the best in every individual fold**. Its overall mean was highest, so the model was selected based on the primary evaluation metric rather than on a single fold.

### Supplementary Logistic Regression Interpretation

Although Random Forest performed best, I also inspected the standardized Logistic Regression coefficients because they are easier to interpret.

| Feature | Mean Standardized Coef |
| :--- | :---: |
| `momentum` | -0.6698 |
| `april_clicks` | -0.3214 |
| `ctr` | +0.2105 |
| `april_impressions` | +0.1852 |
| `weighted_position` | +0.1428 |
| `impressions_total` | +0.1120 |
| `clicks_total` | -0.0984 |
| `feb_clicks` | -0.0762 |
| `active_days` | -0.0410 |

A **positive coefficient** increases the model's predicted decline probability, while a **negative coefficient** decreases it, holding the other standardized features constant.

These coefficients describe the **Logistic Regression model only** and are included as supplementary interpretation, not as evidence of causation.

### Takeaway

Random Forest achieved the best **Precision@50 of 0.4440**, beating the baseline's **0.3920**.

I therefore use **Random Forest as the best learned model** for the error analysis in Section 4.

The purpose of the model is **not to automatically decide which pages should be refreshed**. It is a **prioritization tool to help human reviewers decide which pages deserve attention first**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print('==================================================')
print('SECTION 3: MODEL TRAINING & BASELINE COMPARISON')
print('==================================================')

# 1. Define Week-5 Baseline Rule Score on Eligible Dataset
# Rule: Flag pages where april_clicks < march_clicks, score = april_impressions, else 0
elig_pd['baseline_score'] = np.where(elig_pd['april_clicks'] < elig_pd['march_clicks'], elig_pd['april_impressions'], 0)

# Deterministic Precision@K evaluation function with tie policy
def eval_precision_at_k(df_fold, score_col, k_list=[10, 20, 50, 100]):
    # Primary sort: score_col desc | Secondary tie-break: april_clicks desc | Final tie-break: content_hash_id asc
    sorted_df = df_fold.sort_values(
        by=[score_col, 'april_clicks', 'content_hash_id'],
        ascending=[False, False, True]
    )
    results = {}
    for k in k_list:
        top_k = sorted_df.head(min(k, len(sorted_df)))
        results[f'Precision@{k}'] = float(top_k['decline'].mean()) if len(top_k) > 0 else 0.0
    return results

# 2. Initialize Models
models = {
    'Baseline': None,
    'Logistic Regression': None,
    'Decision Tree': None,
    'Random Forest': None
}

k_eval_list = [10, 20, 50, 100]
fold_results = {name: {f'Precision@{k}': [] for k in k_eval_list} for name in models}
lr_coefs = []

# 3. Execute 5-Fold GroupKFold Cross-Validation
gkf_sec3 = GroupKFold(n_splits=5)
X_sec3 = elig_pd[feature_cols]
y_sec3 = elig_pd['decline']
groups_sec3 = elig_pd['client_hash_id']

for fold, (tr_idx, val_idx) in enumerate(gkf_sec3.split(X_sec3, y_sec3, groups_sec3)):
    tr_df = elig_pd.iloc[tr_idx]
    val_df = elig_pd.iloc[val_idx].copy()

    # Baseline Evaluation
    base_res = eval_precision_at_k(val_df, 'baseline_score', k_eval_list)
    for metric, val in base_res.items():
        fold_results['Baseline'][metric].append(val)

    # Scale numerical features for LR (fit ONLY on training fold)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(tr_df[feature_cols])
    X_val_scaled = scaler.transform(val_df[feature_cols])

    # Logistic Regression
    lr_model = LogisticRegression(random_state=42, max_iter=1000)
    lr_model.fit(X_tr_scaled, tr_df['decline'])
    lr_coefs.append(lr_model.coef_[0])
    val_df['lr_score'] = lr_model.predict_proba(X_val_scaled)[:, 1]
    lr_res = eval_precision_at_k(val_df, 'lr_score', k_eval_list)
    for metric, val in lr_res.items():
        fold_results['Logistic Regression'][metric].append(val)

    # Decision Tree
    dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_model.fit(tr_df[feature_cols], tr_df['decline'])
    val_df['dt_score'] = dt_model.predict_proba(val_df[feature_cols])[:, 1]
    dt_res = eval_precision_at_k(val_df, 'dt_score', k_eval_list)
    for metric, val in dt_res.items():
        fold_results['Decision Tree'][metric].append(val)

    # Random Forest
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(tr_df[feature_cols], tr_df['decline'])
    val_df['rf_score'] = rf_model.predict_proba(val_df[feature_cols])[:, 1]
    rf_res = eval_precision_at_k(val_df, 'rf_score', k_eval_list)
    for metric, val in rf_res.items():
        fold_results['Random Forest'][metric].append(val)

# 4. Print Fold-by-Fold Precision@50 Results
print('\n--- FOLD-BY-FOLD PRECISION@50 RESULTS ---')
fold_p50_df = pd.DataFrame({
    name: fold_results[name]['Precision@50'] for name in models
})
fold_p50_df.index = [f'Fold {i}' for i in range(5)]
print(fold_p50_df.to_string())

# 5. Build & Print Summary Comparison Table
summary_rows = []
for name in models:
    row = {'Model': name}
    for k in k_eval_list:
        metric_key = f'Precision@{k}'
        row[metric_key] = round(float(np.mean(fold_results[name][metric_key])), 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print('\n==================================================')
print('FINAL MODEL VS BASELINE COMPARISON TABLE')
print('==================================================')
print(summary_df.to_string(index=False))

best_p50_model = summary_df.sort_values('Precision@50', ascending=False).iloc[0]['Model']
best_p50_score = summary_df.sort_values('Precision@50', ascending=False).iloc[0]['Precision@50']
base_p50_score = summary_df[summary_df['Model'] == 'Baseline']['Precision@50'].values[0]

print(f'\n⭐ Best Performing Model at Precision@50: {best_p50_model} (Precision@50 = {best_p50_score:.4f} vs Baseline = {base_p50_score:.4f})')

# 6. Standardized Logistic Regression Feature Coefficients
lr_coef_matrix = pd.DataFrame(lr_coefs, columns=feature_cols)
coef_summary = pd.DataFrame({
    'Feature': feature_cols,
    'Mean_Standardized_Coef': lr_coef_matrix.mean().values,
    'Std_Standardized_Coef': lr_coef_matrix.std().values
}).sort_values(by='Mean_Standardized_Coef', key=abs, ascending=False)

print('\n==================================================')
print('LOGISTIC REGRESSION FEATURE COEFFICIENTS (STANDARDIZED Across 5 Folds)')
print('==================================================')
print(coef_summary.to_string(index=False))


SECTION 3: MODEL TRAINING & BASELINE COMPARISON

--- FOLD-BY-FOLD PRECISION@50 RESULTS ---
        Baseline  Logistic Regression  Decision Tree  Random Forest
Fold 0      0.32                 0.32           0.30           0.60
Fold 1      0.20                 0.54           0.20           0.40
Fold 2      0.40                 0.30           0.28           0.52
Fold 3      0.40                 0.38           0.40           0.20
Fold 4      0.64                 0.58           0.44           0.50

FINAL MODEL VS BASELINE COMPARISON TABLE
              Model  Precision@10  Precision@20  Precision@50  Precision@100
           Baseline          0.40          0.37         0.392          0.388
Logistic Regression          0.38          0.37         0.424          0.438
      Decision Tree          0.40          0.39         0.324          0.346
      Random Forest          0.46          0.43         0.444          0.448

⭐ Best Performing Model at Precision@50: Random Forest (Precision@50 = 0.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Out-of-Fold Error Analysis

The **Random Forest** was the best model at Precision@50, so I used its **out-of-fold (OOF) predictions** for this analysis. This means the predictions were made on validation pages that were not used to train the model.

The main goal is still **ranking pages for human content review**, so I focused on Top-K results and also used a **0.5 probability threshold** for supplementary TP/FP/TN/FN analysis.

### What the errors show

I compared the error groups using only the **9 pre-May features**:

`impressions_total`, `clicks_total`, `april_impressions`, `april_clicks`, `feb_clicks`, `momentum`, `ctr`, `active_days`, and `weighted_position`.

- **False Positives (FP):** These pages were associated with **lower pre-May momentum and poorer search positions**. The model ranked some of these pages as higher risk, but they did not decline in May.
- **False Negatives (FN):** These pages were more commonly associated with **higher pre-May momentum and stronger traffic levels**. Their historical performance looked relatively healthy, but some still experienced a decline in May.

These patterns are **observed associations, not evidence that the features caused the errors**.

### Top-50 Review Errors

Random Forest achieved a mean **Precision@50 of 0.4440** across the five validation folds.

This corresponds to approximately **22 correctly prioritized declining pages out of every 50 reviewed**. The remaining pages in the Top 50 were non-declining pages, showing that the model can still produce false positives.

### Business Meaning

- **False Positives:** Human reviewers may spend time checking pages that ultimately did not decline.
- **False Negatives:** Some genuinely declining pages may be missed by the review queue.
- Since review capacity is limited, the goal is not perfect prediction but **better prioritization of pages for human review**.

### What I Learned

1. **What the model handles well:**  
   Random Forest can combine multiple historical signals instead of relying on a single rule.

2. **Where it struggles:**  
   Pages with positive historical momentum can still experience future declines that are difficult to identify using only pre-May information.

3. **What could improve future versions:**  
   Additional leakage-safe signals such as **content freshness, search intent, and pre-May position volatility** could potentially provide more information.

4. **Why human review is still necessary:**  
   The model prioritizes pages using historical performance data, but human reviewers still need to evaluate **content quality, search intent, and whether a refresh is actually appropriate**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print('==================================================')
print('SECTION 4: OUT-OF-FOLD ERROR ANALYSIS & INTERPRETATION')
print('==================================================')

# 1. Re-generate / Extract Out-of-Fold (OOF) Validation Predictions across 5 GroupKFold Splits
gkf_sec4 = GroupKFold(n_splits=5)
X_sec4 = elig_pd[feature_cols]
y_sec4 = elig_pd['decline']
groups_sec4 = elig_pd['client_hash_id']

# Determine the actual best performing model from Section 3 results
# Fits candidates and collects OOF scores deterministically
oof_list = []
for fold, (tr_idx, val_idx) in enumerate(gkf_sec4.split(X_sec4, y_sec4, groups_sec4)):
    tr_df = elig_pd.iloc[tr_idx]
    val_df = elig_pd.iloc[val_idx].copy()
    val_df['fold'] = fold

    # Standardize features using training fold statistics ONLY
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(tr_df[feature_cols])
    X_val_scaled = scaler.transform(val_df[feature_cols])

    # Baseline
    val_df['baseline_score'] = np.where(val_df['april_clicks'] < val_df['march_clicks'], val_df['april_impressions'], 0)

    # Logistic Regression
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_tr_scaled, tr_df['decline'])
    val_df['lr_score'] = lr.predict_proba(X_val_scaled)[:, 1]

    # Decision Tree
    dt = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt.fit(tr_df[feature_cols], tr_df['decline'])
    val_df['dt_score'] = dt.predict_proba(val_df[feature_cols])[:, 1]

    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(tr_df[feature_cols], tr_df['decline'])
    val_df['rf_score'] = rf.predict_proba(val_df[feature_cols])[:, 1]

    oof_list.append(val_df)

oof_pd = pd.concat(oof_list, ignore_index=True)

# Determine model scores for selection
def eval_p50(score_col):
    p50_list = []
    for f in range(5):
        f_df = oof_pd[oof_pd['fold'] == f].sort_values(by=[score_col, 'april_clicks', 'content_hash_id'], ascending=[False, False, True]).head(50)
        p50_list.append(f_df['decline'].mean())
    return np.mean(p50_list)

p50_scores = {
    'Baseline': eval_p50('baseline_score'),
    'Logistic Regression': eval_p50('lr_score'),
    'Decision Tree': eval_p50('dt_score'),
    'Random Forest': eval_p50('rf_score')
}

best_model_name = max(p50_scores, key=p50_scores.get)
score_col_map = {
    'Baseline': 'baseline_score',
    'Logistic Regression': 'lr_score',
    'Decision Tree': 'dt_score',
    'Random Forest': 'rf_score'
}
best_score_col = score_col_map[best_model_name]

print(f'Actual Best Model Selected at Precision@50: {best_model_name} (P@50 = {p50_scores[best_model_name]:.4f})')

# 2. Classify Out-of-Fold Error Groups for Best Model at Classification Threshold = 0.5
oof_pd['best_pred_50'] = (oof_pd[best_score_col] >= 0.5).astype(int)

def get_error_category(row):
    if row['best_pred_50'] == 1 and row['decline'] == 1:
        return 'True Positive (TP)'
    elif row['best_pred_50'] == 1 and row['decline'] == 0:
        return 'False Positive (FP)'
    elif row['best_pred_50'] == 0 and row['decline'] == 0:
        return 'True Negative (TN)'
    else:
        return 'False Negative (FN)'

oof_pd['error_group'] = oof_pd.apply(get_error_category, axis=1)

print(f'\n1. Out-of-Fold Binary Classification Breakdown for {best_model_name} (Threshold = 0.5):')
error_counts = oof_pd['error_group'].value_counts()
error_pcts = (oof_pd['error_group'].value_counts(normalize=True) * 100).round(2)
error_summary = pd.DataFrame({'Count': error_counts, 'Percentage (%)': error_pcts})
print(error_summary.to_string())

# 3. Pre-May Feature Medians across Error Groups vs Overall Population
print(f'\n2. Pre-May Feature Median Profiles across Out-of-Fold Error Groups ({best_model_name}):')
medians_table = oof_pd.groupby('error_group')[feature_cols].median().T
medians_table['Overall_Population'] = oof_pd[feature_cols].median()
print(medians_table.round(4).to_string())

# 4. Top-50 Review Priority Bucket Accuracy per Fold
print(f'\n3. Top-50 Review Priority Bucket Breakdown per Fold ({best_model_name} vs Baseline):')
top50_summary_rows = []
for fold_idx in range(5):
    f_df = oof_pd[oof_pd['fold'] == fold_idx].copy()

    # Best Model Top-50
    best_top50 = f_df.sort_values(by=[best_score_col, 'april_clicks', 'content_hash_id'], ascending=[False, False, True]).head(50)
    best_tp = (best_top50['decline'] == 1).sum()
    best_fp = (best_top50['decline'] == 0).sum()

    # Baseline Top-50
    base_top50 = f_df.sort_values(by=['baseline_score', 'april_clicks', 'content_hash_id'], ascending=[False, False, True]).head(50)
    base_tp = (base_top50['decline'] == 1).sum()
    base_fp = (base_top50['decline'] == 0).sum()

    top50_summary_rows.append({
        'Fold': f'Fold {fold_idx}',
        'BestModel_TP': best_tp, 'BestModel_FP': best_fp, 'BestModel_P@50': f'{best_tp/50:.4f}',
        'Base_TP': base_tp, 'Base_FP': base_fp, 'Base_P@50': f'{base_tp/50:.4f}'
    })

top50_summary_df = pd.DataFrame(top50_summary_rows)
print(top50_summary_df.to_string(index=False))

# 5. Extract Actual Out-of-Fold Prediction Rows (Representative Examples)
print(f'\n4. Actual Out-of-Fold Validation Examples for {best_model_name} (Extracted directly from OOF predictions):')
fp_examples = oof_pd[oof_pd['error_group'] == 'False Positive (FP)'].sort_values(best_score_col, ascending=False).head(2)
fn_examples = oof_pd[oof_pd['error_group'] == 'False Negative (FN)'].sort_values(best_score_col, ascending=True).head(2)

print(f'\n--- Actual OOF False Positive Rows ({best_model_name}: Predicted High Risk, Actual May Decline = 0) ---')
for idx, row in fp_examples.iterrows():
    print(f"Content Hash ID: {row['content_hash_id']} | Fold: {row['fold']} | Score: {row[best_score_col]:.4f} | Target: {row['decline']}")
    print(f"   Features -> Momentum: {row['momentum']:.4f} | Apr Clicks: {row['april_clicks']} | Feb Clicks: {row['feb_clicks']} | Position: {row['weighted_position']:.2f} | CTR: {row['ctr']:.2f}%")

print(f'\n--- Actual OOF False Negative Rows ({best_model_name}: Predicted Low Risk, Actual May Decline = 1) ---')
for idx, row in fn_examples.iterrows():
    print(f"Content Hash ID: {row['content_hash_id']} | Fold: {row['fold']} | Score: {row[best_score_col]:.4f} | Target: {row['decline']}")
    print(f"   Features -> Momentum: {row['momentum']:.4f} | Apr Clicks: {row['april_clicks']} | Feb Clicks: {row['feb_clicks']} | Position: {row['weighted_position']:.2f} | CTR: {row['ctr']:.2f}%")


SECTION 4: OUT-OF-FOLD ERROR ANALYSIS & INTERPRETATION

1. Out-of-Fold Binary Classification Breakdown (Threshold = 0.5):
                     Count  Percentage (%)
error_group                               
True Negative (TN)    7440           45.06
False Negative (FN)   5535           33.52
False Positive (FP)   2200           13.32
True Positive (TP)    1338            8.10

2. Pre-May Feature Median Profiles across Out-of-Fold Error Groups:
error_group        False Negative (FN)  False Positive (FP)  True Negative (TN)  True Positive (TP)  Overall_Population
impressions_total           16740.0000           18472.5000          13990.5000          15401.5000          15657.0000
clicks_total                   68.0000              53.0000             64.0000             49.0000             61.0000
april_impressions            6954.0000            6882.0000           6127.0000           5497.5000           6423.0000
april_clicks                   25.0000              19.0000            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.